In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import os

# Settings 
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('data/raw/ee.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   relative_compactness  768 non-null    float64
 1   surface_area          768 non-null    float64
 2   wall_area             768 non-null    float64
 3   roof_area             768 non-null    float64
 4   overall_height        768 non-null    float64
 5   orientation           768 non-null    int64  
 6   glazing_area          768 non-null    float64
 7   glazing_area_dist     768 non-null    int64  
 8   heating_load          768 non-null    float64
 9   cooling_load          768 non-null    float64
dtypes: float64(8), int64(2)
memory usage: 60.1 KB


# 1. Load Data (10 pts)
#### How many rows does your raw dataset have?
768.
#### What column is your target variable?
heating_load and cooling_load, although I think using just one might be fine as from our correlations observed they were incredibly similar.
#### What issues from your Notebook 01 will you address in this notebook?
Aforementioned collinearity.

# 2. Drop Rows with Missing TARGET (15 pts)
#### How many rows had a missing target value? 
Zero.
#### What percentage of the dataset does that represent?
0.0%
#### How many rows remain after dropping?
768.

In [7]:
# Showing the lack of missing values in the dataset
df.isnull().sum()

relative_compactness    0
surface_area            0
wall_area               0
roof_area               0
overall_height          0
orientation             0
glazing_area            0
glazing_area_dist       0
heating_load            0
cooling_load            0
dtype: int64

# 3. Simple Cleaning (10 pts)
#### Note what was removed (duplicate rows, data-leakage columns w/ explanation, impossible values)  

Minimal cleaning required. None of the rows are duplicates, no impossible variables, etc.  

Dropped cooling_load due to being collinear with heating_load and heating_load having a slightly smoother distribution.

One hot encoded orientation and glazing area due to them being categorical variables (moved to after train/test split)

In [10]:
# Drop cooling_load due to colinearity
df_dropped = df.drop(columns=['cooling_load'])

# 4 Train-Test Split (10 pts)
#### Print shapes of X_train, X_test, y_train, y_test
#### How many rows in training and test sets?
X_training: 614 rows  
X_test: 154 rows  
y_train: 614 rows  
y_test: 154 rows  

In [13]:
from sklearn.model_selection import train_test_split

# Features and target
X = df_dropped.drop(columns=['heating_load'])
y = df_dropped['heating_load']

# Split (standard 80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Print shapes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (614, 8)
X_test: (154, 8)
y_train: (614,)
y_test: (154,)


In [15]:
X.head()

,relative_compactness,surface_area,wall_area,roof_area,overall_height,orientation,glazing_area,glazing_area_dist
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0


In [17]:
y.head()

0    15.55
1    15.55
2    15.55
3    15.55
4    20.84
Name: heating_load, dtype: float64

# 5. Handle Missing Values (25 pts)
There are none to begin with, thankfully.

# 6. Handle Outliers (30 pts)


In [21]:
train_df = X_train.copy()
train_df["heating_load"] = y_train.values

mask = pd.Series(True, index=train_df.index)

for col in X_train.columns:
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    mask = mask & train_df[col].between(lower, upper)

train_df_clean = train_df[mask]

X_train_clean = train_df_clean.drop(columns=["heating_load"])
y_train_clean = train_df_clean["heating_load"]

In [23]:
X_train.describe()

,relative_compactness,surface_area,wall_area,roof_area,overall_height,orientation,glazing_area,glazing_area_dist
count,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000
mean,0.761678,673.829805,318.819218,177.505293,5.221498,3.511401,0.235423,2.799674
std,0.105424,88.161209,43.335953,44.960238,1.751195,1.108635,0.134152,1.551908
min,0.620000,514.500000,245.000000,110.250000,3.500000,2.000000,0.000000,0.000000
25%,0.660000,612.500000,294.000000,147.000000,3.500000,3.000000,0.100000,1.250000
50%,0.740000,686.000000,318.500000,220.500000,3.500000,4.000000,0.250000,3.000000
75%,0.820000,759.500000,343.000000,220.500000,7.000000,4.000000,0.400000,4.000000
max,0.980000,808.500000,416.500000,220.500000,7.000000,5.000000,0.400000,5.000000


# Key Findings Summary
#### Finding 1
There are no missing values or duplicates.
#### Finding 2
Dropped cooling_load target due to collinearity with heating_load.
#### Finding 3
One Hot encoded orientation and glazing_area_dist.
#### Finding 4
Removed outliers in training data.

### Save Notebooks

In [27]:
# Create output directory if it doesn't exist
import os
os.makedirs('data/processed', exist_ok=True)

#Save datasets
X_train.to_csv('data/processed/X_train.csv', index=False)
X_test.to_csv('data/processed/X_test.csv', index=False)
y_train.to_csv('data/processed/y_train.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)
df_dropped.to_csv('data/processed/df_dropped.csv', index=False)

print("Files saved successfully.")

Files saved successfully.
